# Household Income Contribution & Satisfaction Among UK Parents
### A Quantitative Analysis Using Understanding Society (Wave 14, 2023/2024)
 
**Author:** Evangelos Pourikas
 
This notebook recreates in Python an analysis originally conducted in SPSS 
for a BSc dissertation at City, St George's, University of London.

 **Research Questions:**
 1. What is the relationship between the proportion of personal income that parents 
    contribute to total household income and their satisfaction with it?
 2. Does being a mother or a father change this relationship?
 3. Which other socio-economic factors affect this relationship?

---
## Section 1: Setup & Data Preparation

In [ ]:
# Install pyreadstat if needed (uncomment the line below)
#!pip install pyreadstat

import pandas as pd
import numpy as np
import pyreadstat
import os
from scipy import stats
from scipy.stats import chi2_contingency, pearsonr
import statsmodels.api as sm
from statsmodels.formula.api import logit
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

### 1.1 Load the Data
 
 We need two files from Understanding Society Wave 14:
 - `n_indresp.sav` — Individual-level responses
 - `n_hhresp.sav` — Household-level responses
 
 **Note:** Update the file paths below to match your local data location.
 The data can be downloaded from the [UK Data Service](https://ukdataservice.ac.uk/) (SN: 6614).

In [ ]:
#UPDATE THESE PATHS to where your data files are stored
INDRESP_PATH = 'data/n_indresp.sav'
HHRESP_PATH = 'data/n_hhresp.sav'

# Load individual response file
ind_df, ind_meta = pyreadstat.read_sav(INDRESP_PATH)
print(f"Individual responses loaded: {ind_df.shape[0]} rows, {ind_df.shape[1]} columns")

# Load household response file
hh_df, hh_meta = pyreadstat.read_sav(HHRESP_PATH)
print(f"Household responses loaded: {hh_df.shape[0]} rows, {hh_df.shape[1]} columns")

### 1.2 Merge Individual and Household Files
 
 We merge on `n_hidp` (household identifier) to combine individual-level 
 variables (satisfaction, gender, income, etc.) with household-level variables 
 (total household income).

In [ ]:
# Keep only the variables we need from each file
ind_vars = [
    'pidp', 'n_hidp', 'n_inding2_xw',  # identifiers and weight
    'n_sex', 'n_dvage', 'n_hhtype_dv',  # demographics
    'n_sclfsat2',  # satisfaction with household income (dependent variable)
    'n_scsf1',  # general health
    'n_jbstat',  # job status
    'n_howlng',  # hours of housework per week
    'n_fimnnet_dv',  # personal net monthly income
]

hh_vars = [
    'n_hidp',
    'n_fihhmnnet1_dv',  # total net household monthly income
    'n_agechy_dv',  # age of youngest child in household
]

ind_subset = ind_df[ind_vars].copy()
hh_subset = hh_df[hh_vars].copy()

# Merge on household identifier
df = pd.merge(ind_subset, hh_subset, on='n_hidp', how='left')
print(f"Merged dataset: {df.shape[0]} rows, {df.shape[1]} columns")

### 1.3 Filter to Couple Households with Children Under 16
 
 Household types 10, 11, 12 represent couple households (married, cohabiting, 
 civil partnership). We also filter for households where the youngest child 
 is under 16.

In [ ]:
#Filter to couple households with children under 16
df = df[
    (df['n_hhtype_dv'].isin([10, 11, 12])) & 
    (df['n_agechy_dv'] >= 0) & 
    (df['n_agechy_dv'] < 16)
].copy()

print(f"After filtering to couple households with children under 16: {df.shape[0]} rows")

### 1.4 Handle Missing Values
 
 In Understanding Society, negative values indicate missing data:
 -1 = Don't know, -2 = Refused, -7 = Proxy, -8 = Inapplicable, -9 = Missing.
 We replace all negative values with NaN.

In [ ]:
#Variables to clean
vars_to_clean = ['n_sclfsat2', 'n_sex', 'n_howlng', 'n_agechy_dv', 
                 'n_jbstat', 'n_scsf1', 'n_fimnnet_dv', 'n_fihhmnnet1_dv']

for var in vars_to_clean:
    df.loc[df[var] < 0, var] = np.nan

# Report missing values
print("Missing values after cleaning:")
print(df[vars_to_clean].isnull().sum())
print(f"\nTotal rows remaining: {df.shape[0]}")


---
## Section 2: Variable Creation & Recoding

 ### 2.1 Key Independent Variable: Proportion Contributed to Household Income
 
 We compute the percentage each individual contributes to their total household 
 income: `(personal net income / household net income) * 100`

In [ ]:
#Compute percentage contributed
df['pct_contributed'] = (df['n_fimnnet_dv'] / df['n_fihhmnnet1_dv']) * 100

# Recode into 6 categories
bins = [-np.inf, 0.9999, 25, 50, 75, 99.9999, np.inf]
labels = ['No Contribution', '1%-25%', '25%-50%', '50%-75%', '75%-99%', 'Full Contribution']
df['pct_contributed_cat'] = pd.cut(df['pct_contributed'], bins=bins, labels=labels)

print("Distribution of contribution categories:")
print(df['pct_contributed_cat'].value_counts().sort_index())

### 2.2 Dependent Variable: Satisfaction with Household Income
 
 The original scale runs from 1 (Completely Dissatisfied) to 7 (Completely Satisfied).
 For the logistic regression, we recode into binary:
 - Dissatisfied = 1, 2, 3
 - Satisfied = 5, 6, 7
 - Neither (4) = excluded

In [ ]:
#Keep the original 1-7 scale for bivariate analysis
df['satisfaction_scale'] = df['n_sclfsat2'].copy()

# Create binary version for logistic regression
df['satisfaction_binary'] = np.nan
df.loc[df['n_sclfsat2'].isin([1, 2, 3]), 'satisfaction_binary'] = 0  # Dissatisfied
df.loc[df['n_sclfsat2'].isin([5, 6, 7]), 'satisfaction_binary'] = 1  # Satisfied
# 4 (Neither) is left as NaN and will be excluded

print("Binary satisfaction distribution:")
print(df['satisfaction_binary'].value_counts().rename({0: 'Dissatisfied', 1: 'Satisfied'}))

### 2.3 Recode Household Income into Deciles


In [ ]:
# Calculate weighted decile boundaries
df_valid_income = df.dropna(subset=['n_fihhmnnet1_dv', 'n_inding2_xw'])

# Use numpy weighted percentiles
from statsmodels.stats.weightstats import DescrStatsW
weighted_stats = DescrStatsW(df_valid_income['n_fihhmnnet1_dv'], 
                              weights=df_valid_income['n_inding2_xw'])

# Extract each boundary as a float
boundaries = []
for p in [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]:
    val = weighted_stats.quantile(p/100, return_pandas=False)
    if hasattr(val, '__len__'):
        boundaries.append(float(val[0]))
    else:
        boundaries.append(float(val))

income_bins = [-np.inf] + boundaries[1:-1] + [np.inf]
income_labels = [
    'Lowest to £1997', '£1998 to £2678', '£2679 to £3120',
    '£3121 to £3576', '£3577 to £4014', '£4015 to £4562',
    '£4563 to £5172', '£5173 to £5908', '£5909 to £7246',
    '£7247 to Highest'
]

df['income_decile'] = pd.cut(df['n_fihhmnnet1_dv'], bins=income_bins, labels=income_labels)

print("Household income decile distribution:")
print(df['income_decile'].value_counts().sort_index())

### 2.4 Recode Job Status

In [ ]:
#Recode job status — keep categories 1-8, exclude 10+
job_labels = {
    1: 'Self Employed',
    2: 'Paid Employment (ft/pt)',
    3: 'Unemployed',
    4: 'Retired',
    5: 'On Maternity Leave',
    6: 'Family Care or Home',
    7: 'Full-Time Student',
    8: 'LT Sick or Disabled'
}

df['job_status'] = df['n_jbstat'].map(job_labels)
# Anything not mapped (categories 10+) becomes NaN automatically

print("Job status distribution:")
print(df['job_status'].value_counts())

### 2.5 Recode Gender and General Health

In [ ]:
# Gender
df['gender'] = df['n_sex'].map({1: 'Male', 2: 'Female'})

# General health
health_labels = {1: 'Excellent', 2: 'Very Good', 3: 'Good', 4: 'Fair', 5: 'Poor'}
df['general_health'] = df['n_scsf1'].map(health_labels)

print("Gender distribution:")
print(df['gender'].value_counts())
print("\nGeneral health distribution:")
print(df['general_health'].value_counts())

---
 ## Section 3: Bivariate Analysis

### Helper Function: Chi-Square Test with Cramér's V

In [ ]:
def chi_square_test(df, var1, var2, weight_col='n_inding2_xw'):
    """
    Perform a weighted chi-square test and calculate Cramér's V.
    """
    # Create weighted crosstab
    ct = pd.crosstab(df[var1], df[var2], values=df[weight_col], aggfunc='sum')
    
    # Chi-square test
    chi2, p, dof, expected = chi2_contingency(ct)
    
    # Cramér's V
    n = ct.sum().sum()
    min_dim = min(ct.shape[0], ct.shape[1]) - 1
    cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
    
    print(f"Chi-Square Test: {var1} × {var2}")
    print(f"  χ² = {chi2:.3f}")
    print(f"  p-value = {p:.6f} {'(significant)' if p < 0.05 else '(not significant)'}")
    print(f"  Cramér's V = {cramers_v:.3f}")
    print(f"  df = {dof}")
    print()
    
    return ct, chi2, p, cramers_v

### Helper Function: Stacked Bar Chart

In [ ]:
def stacked_bar_chart(df, group_var, satisfaction_var, weight_col, title, xlabel):
    """
    Create a stacked percentage bar chart of satisfaction levels by group.
    """
    # Create weighted crosstab with percentages
    ct = pd.crosstab(df[group_var], df[satisfaction_var], 
                      values=df[weight_col], aggfunc='sum', normalize='index') * 100
    
    # Define satisfaction colours (matching SPSS output)
    colors = ['#009900', '#66CC00', '#99FF66', '#C0C0C0', '#FF9933', '#FF3300', '#CC0000']
    
    # Satisfaction labels
    sat_labels = {
        1.0: 'Completely Satisfied',
        2.0: 'Mostly Satisfied', 
        3.0: 'Somewhat Satisfied',
        4.0: 'Neither',
        5.0: 'Somewhat Dissatisfied',
        6.0: 'Mostly Dissatisfied',
        7.0: 'Completely Dissatisfied'
    }
    
    ct = ct.rename(columns=sat_labels)
    
    # Sort columns in satisfaction order
    col_order = ['Completely Satisfied', 'Mostly Satisfied', 'Somewhat Satisfied',
                 'Neither', 'Somewhat Dissatisfied', 'Mostly Dissatisfied', 
                 'Completely Dissatisfied']
    ct = ct[[c for c in col_order if c in ct.columns]]
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 7))
    ct.plot(kind='bar', stacked=True, color=colors[:len(ct.columns)], ax=ax, width=0.7)
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel('Percent', fontsize=12)
    ax.set_ylim(0, 100)
    ax.legend(title='Satisfaction with Household Income', 
              bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    
    # Add percentage labels
    for container_idx, container in enumerate(ax.containers):
        for bar in container:
            height = bar.get_height()
            if height > 3:  # Only label if segment is big enough
                ax.text(bar.get_x() + bar.get_width()/2, 
                        bar.get_y() + height/2,
                        f'{height:.0f}%', ha='center', va='center', 
                        fontsize=8, fontweight='bold', color='white')
    
    plt.tight_layout()
    plt.savefig(f'outputs/figures/{group_var}_by_satisfaction.png', dpi=150, bbox_inches='tight')
    plt.show()

### 3.1 Contribution to Household Income × Satisfaction
 
 This is the core relationship of the study. We test whether the proportion 
 contributed to household income is significantly associated with satisfaction.


In [ ]:
# Drop rows with missing values for this analysis
df_valid = df.dropna(subset=['pct_contributed_cat', 'satisfaction_scale', 'n_inding2_xw'])

# Chi-square test
ct, chi2, p, v = chi_square_test(df_valid, 'pct_contributed_cat', 'satisfaction_scale')

# Stacked bar chart
stacked_bar_chart(df_valid, 'pct_contributed_cat', 'satisfaction_scale', 'n_inding2_xw',
                  'Percentage Contributed to Household Income by Satisfaction',
                  '% of Contributed into Total Net Household Income')

### 3.2 Gender × Satisfaction

In [ ]:
df_valid = df.dropna(subset=['gender', 'satisfaction_scale', 'n_inding2_xw'])

# Chi-square test
ct, chi2, p, v = chi_square_test(df_valid, 'gender', 'satisfaction_scale')

### 3.3 Three-Way Crosstabulation: Contribution × Satisfaction × Gender
 
 Although gender alone was not significant, the literature suggests that 
 gender dynamics become visible when interacted with income contribution.
 We focus on those reporting being "Completely Dissatisfied" (1).

In [ ]:
df_valid = df.dropna(subset=['pct_contributed_cat', 'satisfaction_scale', 'gender', 'n_inding2_xw'])

# Three-way crosstab — focusing on Completely Dissatisfied (1)
for sex in ['Male', 'Female']:
    subset = df_valid[df_valid['gender'] == sex]
    ct = pd.crosstab(subset['pct_contributed_cat'], subset['satisfaction_scale'],
                      values=subset['n_inding2_xw'], aggfunc='sum', normalize='index') * 100
    
    # Chi-square test for this gender
    ct_raw = pd.crosstab(subset['pct_contributed_cat'], subset['satisfaction_scale'],
                          values=subset['n_inding2_xw'], aggfunc='sum')
    chi2, p_val, dof, expected = chi2_contingency(ct_raw)
    
    label = 'Fathers' if sex == 'Male' else 'Mothers'
    print(f"\n{label} — Contribution × Satisfaction")
    print(f"  χ² = {chi2:.3f}, p = {p_val:.6f}")
    
    if 1.0 in ct.columns:
        print(f"\n  % Completely Dissatisfied by contribution level:")
        print(ct[1.0].round(1).to_string())

### 3.4 General Health × Satisfaction


In [ ]:
df_valid = df.dropna(subset=['general_health', 'satisfaction_scale', 'n_inding2_xw'])

ct, chi2, p, v = chi_square_test(df_valid, 'general_health', 'satisfaction_scale')

stacked_bar_chart(df_valid, 'general_health', 'satisfaction_scale', 'n_inding2_xw',
                  'General Health by Satisfaction with Household Income',
                  'General Health')

### 3.5 Job Status × Satisfaction

In [ ]:
df_valid = df.dropna(subset=['job_status', 'satisfaction_scale', 'n_inding2_xw'])

ct, chi2, p, v = chi_square_test(df_valid, 'job_status', 'satisfaction_scale')

stacked_bar_chart(df_valid, 'job_status', 'satisfaction_scale', 'n_inding2_xw',
                  'Job Status by Satisfaction with Household Income',
                  'Job Status')

### 3.6 Correlations (Continuous Variables)

In [ ]:
# Household Income × Satisfaction
df_corr = df.dropna(subset=['n_fihhmnnet1_dv', 'satisfaction_scale'])
r, p = pearsonr(df_corr['n_fihhmnnet1_dv'], df_corr['satisfaction_scale'])
print(f"Household Income × Satisfaction: r = {r:.3f}, p = {p:.6f}")

# Hours of Housework × Satisfaction
df_corr = df.dropna(subset=['n_howlng', 'satisfaction_scale'])
r, p = pearsonr(df_corr['n_howlng'], df_corr['satisfaction_scale'])
print(f"Hours of Housework × Satisfaction: r = {r:.3f}, p = {p:.6f}")

---
 ## Section 4: Logistic Regression
 
 We now test whether the relationship between contribution to household income 
 and satisfaction persists when controlling for gender, general health, job status, 
 household income, and housework hours.
 
 **Reference categories:**
 - Contribution: 50%-75%
 - Household Income: £3,577–£4,014 (5th decile)
 - Job Status: Unemployed
 - General Health: Good
 - Gender: Female
 - Housework: Continuous (no reference)

In [ ]:
# Prepare data for logistic regression
df_reg = df.dropna(subset=[
    'satisfaction_binary', 'pct_contributed_cat', 'income_decile',
    'general_health', 'job_status', 'gender', 'n_howlng', 'n_inding2_xw'
]).copy()

# Recode satisfaction: 0 = Satisfied (reference), 1 = Dissatisfied (outcome)
df_reg['dissatisfied'] = (df_reg['satisfaction_binary'] == 0).astype(int)

print(f"Sample size for logistic regression: {df_reg.shape[0]}")
print(f"\nOutcome distribution:")
print(df_reg['dissatisfied'].value_counts().rename({0: 'Satisfied', 1: 'Dissatisfied'}))

In [ ]:
# Set reference categories using C() with Treatment coding
formula = (
    'dissatisfied ~ '
    'C(pct_contributed_cat, Treatment(reference="50%-75%")) + '
    'C(income_decile, Treatment(reference="£3577 to £4014")) + '
    'C(general_health, Treatment(reference="Good")) + '
    'C(job_status, Treatment(reference="Unemployed")) + '
    'C(gender, Treatment(reference="Female")) + '
    'n_howlng'
)

# Fit weighted logistic regression
model = logit(formula, data=df_reg).fit(
    freq_weights=df_reg['n_inding2_xw'],
    disp=0  # Suppress iteration output
)

print(model.summary())

### 4.1 Odds Ratios with 95% Confidence Intervals
 
 Odds ratios (Exp(B)) are easier to interpret than raw coefficients.
 An odds ratio > 1 means higher odds of being dissatisfied compared to the 
 reference category; < 1 means lower odds.

In [ ]:
# Calculate odds ratios and confidence intervals
odds_ratios = np.exp(model.params)
conf_int = np.exp(model.conf_int())
conf_int.columns = ['Lower 95% CI', 'Upper 95% CI']

# Combine into one table
results = pd.DataFrame({
    'Odds Ratio (Exp B)': odds_ratios,
    'Lower 95% CI': conf_int['Lower 95% CI'],
    'Upper 95% CI': conf_int['Upper 95% CI'],
    'p-value': model.pvalues
})

# Drop intercept for cleaner display
results = results.drop('Intercept', errors='ignore')

# Format for display
results['Sig'] = results['p-value'].apply(
    lambda x: '***' if x < 0.001 else ('**' if x < 0.01 else ('*' if x < 0.05 else 'ns'))
)

print("Logistic Regression Results — Predicting Dissatisfaction with Household Income")
print("=" * 90)
print(results.round(3).to_string())

### 4.2 Model Fit Statistics

In [ ]:
# Model chi-square (likelihood ratio test)
llr_stat = model.llr
llr_pvalue = model.llr_pvalue
print(f"Model Chi-Square (LR): {llr_stat:.3f}")
print(f"Model p-value: {llr_pvalue:.6f}")

# Pseudo R-squared (McFadden's — similar concept to Nagelkerke)
print(f"Pseudo R² (McFadden): {model.prsquared:.3f}")

# Log-likelihood
print(f"Log-Likelihood: {model.llf:.3f}")
print(f"AIC: {model.aic:.3f}")
print(f"BIC: {model.bic:.3f}")

---
 ## Summary of Key Findings
 
 1. **Non-contributors and full contributors** to household income are the 
    most satisfied — dual-earner households report the lowest satisfaction.
 
 2. **Fathers are more likely** to report dissatisfaction with household income 
    compared to mothers, likely shaped by breadwinning social norms.
 
 3. **Mothers contributing 75%+** of household income show the highest rates of 
    complete dissatisfaction — the opposite pattern to fathers.
 
 4. **General health** emerged as the strongest predictor: individuals in poor 
    health were significantly more likely to be dissatisfied, even after 
    controlling for income and employment.
 
 5. These findings **contradict** a similar Czech study (Mysíková, 2016) which 
    found no significant effects among couples with children.